# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

In [1]:
"""
Method: Pairwise ranking via Logistic Regression on feature differences

There is no ground-truth label in this problem. The Week 4 baseline (gates, scoring signals) is itself a hand-written scoring rule built  based on each tier not a target to predict against. So this isn't classification or regression toward a known truth; it's a comparison between two different ways of scoring and ranking the same rows.

What actually matters for this problem is the order pages fall in, not the exact score value — the baseline's real job is to rank pages by refresh urgency. Pairwise ranking directly optimizes for "does page A outrank page B," which matches that goal more precisely than trying to hit an arbitrary numeric score.

To keep the comparison fair, the model uses the same signals the baseline formula uses no additional features, no leakage.

Logistic Regression is used because it's the simplest model that can learn. The baseline combines them multiplicatively with fixed, hand-picked weighting signals. Training a Logistic Regression on pairwise feature differences tests whether a different, learned combination of the same signals produces a meaningfully different — and possibly more sensible — ranking than the fixed formula.
"""

'\nMethod: Pairwise ranking via Logistic Regression on feature differences\n\nThere is no ground-truth label in this problem. The Week 4 baseline (gates, scoring signals) is itself a hand-written scoring rule built  based on each tier not a target to predict against. So this isn\'t classification or regression toward a known truth; it\'s a comparison between two different ways of scoring and ranking the same rows.\n\nWhat actually matters for this problem is the order pages fall in, not the exact score value — the baseline\'s real job is to rank pages by refresh urgency. Pairwise ranking directly optimizes for "does page A outrank page B," which matches that goal more precisely than trying to hit an arbitrary numeric score.\n\nTo keep the comparison fair, the model uses the same signals the baseline formula uses no additional features, no leakage.\n\nLogistic Regression is used because it\'s the simplest model that can learn. The baseline combines them multiplicatively with fixed, hand

## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

In [2]:
"""
Grouped by client, not time-aware.

Each content_hash_id appears exactly once in the dataset (verified: value_counts().max() == 1), so there is no repeated time series at the row level to be time-aware about. The Week 4 aggregation already collapsed the daily-grain source data into one summary row per page, with trend_pct and gsc_sum_position computed across the full time window per page. A time-aware split would be guarding against a leak that structurally cannot occur here, so it isn't used.

The real leak risk is client-level: multiple pages likely share the same client_hash_id, and if pages are split randomly, pages from the same client could land on both sides of train/test. Client-level effects (e.g. one client's whole site trending down for reasons unrelated to trend_pct or gsc_sum_position individually) could let the model partly learn "this client's pages behave a certain way" instead of the actual signal relationship being tested. Splitting by client_hash_id, so every page belonging to a given client stays entirely on one side, closes that leak.

Because this is a pairwise ranking setup, the split happens at the row level first, before pairs are generated: clients are divided into a train pool and a test pool, and only afterward are pairs sampled — separately — within each pool. This guarantees no single row, and no client, appears on both sides of any pair.
"""

'\nGrouped by client, not time-aware.\n\nEach content_hash_id appears exactly once in the dataset (verified: value_counts().max() == 1), so there is no repeated time series at the row level to be time-aware about. The Week 4 aggregation already collapsed the daily-grain source data into one summary row per page, with trend_pct and gsc_sum_position computed across the full time window per page. A time-aware split would be guarding against a leak that structurally cannot occur here, so it isn\'t used.\n\nThe real leak risk is client-level: multiple pages likely share the same client_hash_id, and if pages are split randomly, pages from the same client could land on both sides of train/test. Client-level effects (e.g. one client\'s whole site trending down for reasons unrelated to trend_pct or gsc_sum_position individually) could let the model partly learn "this client\'s pages behave a certain way" instead of the actual signal relationship being tested. Splitting by client_hash_id, so eve

## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
from google.colab import userdata
auth=userdata.get("HF_TOKEN")
import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from scipy.stats import spearmanr
con=duckdb.connect()
con.sql("INSTALL httpfs;")
con.sql("LOAD httpfs;")
con.sql(f"""
CREATE SECRET (
    TYPE huggingface,
    TOKEN '{auth}'
);
""")

┌─────────┐
│ Success │
│ boolean │
├─────────┤
│ true    │
└─────────┘

In [4]:
df = con.sql(f"""
SELECT *
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [5]:
dfF = con.sql(f"""
SELECT SUM(gsc_impressions) AS gsc_impressions, client_hash_id, content_hash_id
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-02/data_0.parquet'
GROUP BY client_hash_id, content_hash_id
""").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

In [6]:
dfM = con.sql(f"""
SELECT SUM(gsc_impressions) AS gsc_impressions, client_hash_id, content_hash_id
FROM 'hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet'
GROUP BY client_hash_id, content_hash_id
""").df()

In [7]:
df['ctr'] = df['gsc_clicks'] / df['gsc_impressions']
df['avg_engagement_sec_per_session'] = df['ga4_total_engagement_sec'] / df['ga4_sessions']
df['scroll_rate'] = df['scroll_events'] / df['ga4_pageviews']

for col in ['ctr', 'avg_engagement_sec_per_session', 'scroll_rate']:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan)
    df[col] = df[col].astype('float64')

print(df['ctr'].dtype)
print(df['avg_engagement_sec_per_session'].dtype)
print(df['scroll_rate'].dtype)

float64
float64
float64


In [8]:
df_trend = dfM.merge(dfF, on=['client_hash_id', 'content_hash_id'], suffixes=('_feb', '_mar'), how='outer')

In [9]:
df_trend = df_trend[df_trend['gsc_impressions_feb'] >= 30]
df_trend = df_trend[df_trend['gsc_impressions_mar'] > 0]

df_trend['trend_pct'] = (
    (df_trend['gsc_impressions_mar'] - df_trend['gsc_impressions_feb'])
    / df_trend['gsc_impressions_feb']
) * 100

clip_value = df_trend['trend_pct'].quantile(0.99)
df_trend['trend_pct'] = df_trend['trend_pct'].clip(lower=-clip_value, upper=clip_value)

In [10]:
df['ga4_data_available'] = df['ga4_data_available'].fillna(False)

conditions = [
    df['gsc_data_available'] & df['ga4_data_available'],
    df['gsc_data_available'] & ~df['ga4_data_available'],
    ~df['gsc_data_available'] & df['ga4_data_available'],
    ~df['gsc_data_available'] & ~df['ga4_data_available']
]

tiers = ['Full', 'GSC-only', 'GA4-only', 'unavailable']

df['tier'] = np.select(conditions, tiers, default=None)

In [11]:
df = df.groupby(['client_hash_id', 'content_hash_id'], as_index=False).agg(
    gsc_sum_position=('gsc_sum_position', 'sum'),
    gsc_avg_position=('gsc_avg_position', 'mean'),
    gsc_impressions=('gsc_impressions', 'sum'),
    gsc_clicks=('gsc_clicks', 'sum'),
    ctr=('ctr', 'mean'),
    ga4_pageviews=('ga4_pageviews', 'sum'),
    ga4_sessions=('ga4_sessions', 'sum'),
    ga4_users=('ga4_users', 'sum'),
    ga4_engaged_sessions=('ga4_engaged_sessions', 'sum'),
    ga4_total_engagement_sec=('ga4_total_engagement_sec', 'sum'),
    avg_engagement_sec_per_session=('avg_engagement_sec_per_session', 'mean'),
    scroll_events=('scroll_events', 'sum'),
    gsc_data_available=('gsc_data_available', 'first'),
    ga4_data_available=('ga4_data_available', 'first'),
    tier=('tier', 'first'),
    scroll_rate=('scroll_rate', 'mean')
)

In [ ]:
df = df.merge(df_trend[['client_hash_id', 'content_hash_id', 'trend_pct']], on=['client_hash_id', 'content_hash_id'], how='left')

In [ ]:
df_full_tier = df[df['tier'] == 'Full']
conditions = [
    df_full_tier['trend_pct'] < -50,
    (df_full_tier['trend_pct'] >= -50) & (df_full_tier['trend_pct'] < -15),
    (df_full_tier['trend_pct'] >= -15) & (df_full_tier['trend_pct'] <= 15),
    (df_full_tier['trend_pct'] > 15) & (df_full_tier['trend_pct'] <= 50),
    df_full_tier['trend_pct'] > 50
]
ranks = ['5-Sharp decline', '4-Mild decline', '3-Flat', '2-Mild growth', '1-Strong growth']
df_full_tier['trend_dir'] = np.select(conditions, ranks, default=None)

In [ ]:
display(df_full_tier.groupby('trend_dir').agg(
    {
        'gsc_sum_position': ['mean', 'count'],
        'gsc_avg_position': ['mean', 'count'],
        'gsc_impressions' : ['mean', 'count'],
        'gsc_clicks' : ['mean', 'count'],
        'ga4_pageviews' : ['mean', 'count'],
        'ga4_sessions' : ['mean', 'count'],
        'ga4_users' : ['mean', 'count'],
        'ga4_engaged_sessions' : ['mean', 'count'],
        'ctr' : ['mean', 'count'],
        'ga4_total_engagement_sec' : ['mean', 'count'],
        'avg_engagement_sec_per_session' : ['mean', 'count'],
        'scroll_events' : ['mean', 'count'],
        'scroll_rate' : ['mean', 'count']
    }
))

In [ ]:
def build_tier_frames(df):
    df_full = df[df['tier'] == 'Full'].copy()
    df_gsc_only = df[df['tier'] == 'GSC-only'].copy()
    df_ga4_only = df[df['tier'] == 'GA4-only'].copy()

    df_full['ga4_sessions_pct_rank'] = df_full.groupby('client_hash_id')['ga4_sessions'].transform(
        lambda x: x.rank(pct=True)
    )
    df_full['needs_refresh'] = (
        (df_full['trend_pct'] < -15) &
        (df_full['ga4_sessions_pct_rank'] <= 0.15)
    ).astype(int)

    df_gsc_only['needs_refresh'] = (
        (df_gsc_only['trend_pct'] < -15) &
        (df_gsc_only['gsc_avg_position'] > 10)
    ).astype(int)

    df_ga4_only['needs_refresh'] = (
        (df_ga4_only['ga4_sessions'] <= 2) &
        (df_ga4_only['ga4_engaged_sessions'] <= 2)
    ).astype(int)

    return df_full, df_gsc_only, df_ga4_only

In [ ]:
df_full, df_gsc_only, df_ga4_only = build_tier_frames(df)

In [ ]:
display(df_full['needs_refresh'].value_counts())
display(df_gsc_only['needs_refresh'].value_counts())
display(df_ga4_only['needs_refresh'].value_counts())

In [ ]:
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

df_full_eligible = df_full[df_full['needs_refresh'] == 1].copy()

norm_avg_position = normalize(df_full_eligible['gsc_avg_position'])
norm_ctr = normalize(df_full_eligible['ctr'])
norm_clicks = normalize(df_full_eligible['gsc_clicks'])
norm_engaged_sessions = normalize(df_full_eligible['ga4_engaged_sessions'])
norm_scroll = normalize(df_full_eligible['scroll_rate'])
norm_engagement_sec = normalize(df_full_eligible['avg_engagement_sec_per_session'])
norm_pv_users_ratio = normalize(df_full_eligible['ga4_pageviews'] / df_full_eligible['ga4_users'])

df_full_eligible['score'] = (
     0.25 * norm_avg_position +
    -0.15 * norm_ctr +
     0.10 * norm_clicks +
    -0.117 * norm_engaged_sessions +
    -0.117 * norm_scroll +
    -0.117 * norm_engagement_sec +
    -0.15 * norm_pv_users_ratio
)
display(df_full_eligible[['gsc_avg_position', 'ctr', 'gsc_clicks', 'score']].sort_values('score', ascending=False).head(10))

In [ ]:
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())


df_gsc_only_eligible = df_gsc_only[df_gsc_only['needs_refresh'] == 1].copy()

norm_sum_position = normalize(df_gsc_only_eligible['gsc_sum_position'])
norm_ctr = normalize(df_gsc_only_eligible['ctr'])
norm_clicks = normalize(df_gsc_only_eligible['gsc_clicks'])

df_gsc_only_eligible['score'] = (
     0.30 * norm_sum_position +
    -0.50 * norm_ctr +
     0.20 * norm_clicks
)
display(df_gsc_only_eligible[['gsc_sum_position', 'ctr', 'gsc_clicks', 'score']].sort_values('score', ascending=False).head(10))


In [ ]:
def normalize(series):
    return (series - series.min()) / (series.max() - series.min())

df_ga4_only_eligible = df_ga4_only[df_ga4_only['needs_refresh'] == 1].copy()

norm_engaged_sessions = normalize(df_ga4_only_eligible['ga4_engaged_sessions'])
norm_users = normalize(df_ga4_only_eligible['ga4_users'])
norm_engagement_sec = normalize(df_ga4_only_eligible['avg_engagement_sec_per_session'])
norm_pageviews = normalize(df_ga4_only_eligible['ga4_pageviews'])
norm_scroll = normalize(df_ga4_only_eligible['scroll_rate'])

df_ga4_only_eligible['score'] = (
    -0.40 * norm_engaged_sessions +
    -0.20 * norm_users +
    -0.15 * norm_engagement_sec +
    -0.15 * norm_pageviews +
    -0.10 * norm_scroll
)
display(df_ga4_only_eligible[['ga4_engaged_sessions', 'ga4_users', 'ga4_pageviews', 'score']].sort_values('score', ascending=False).head(10))

In [ ]:
def expected_ctr(position):
    if position <= 1: return 0.30
    elif position <= 2: return 0.17
    elif position <= 3: return 0.10
    elif position <= 5: return 0.06
    elif position <= 10: return 0.02
    else: return 0.01


df_full_eligible['expected_ctr'] = df_full_eligible['gsc_avg_position'].apply(expected_ctr)
df_full_eligible['low_ctr_for_position'] = df_full_eligible['ctr'] < (0.5 * df_full_eligible['expected_ctr'])
df_full_eligible['low_position'] = df_full_eligible['gsc_avg_position'] > 10
df_full_eligible['low_engagement'] = (df_full_eligible['ga4_engaged_sessions'] / df_full_eligible['ga4_sessions']) < 0.50
df_full_eligible['low_scroll'] = df_full_eligible['scroll_rate'] == 0
df_full_eligible['low_engagement_time'] = df_full_eligible['avg_engagement_sec_per_session'] == 0

def build_action_text_full(row):
    lines = []
    if row['low_position']:
        lines.append("Ranks below page 1 (position > 10).")
    if row['low_ctr_for_position']:
        lines.append(f"CTR is under half of what's typical for its position (expected ~{row['expected_ctr']:.0%}).")
    if row['low_engagement']:
        lines.append("Engagement rate below 50%.")
    if row['low_scroll']:
        lines.append("Zero recorded clicks despite adequate impressions — CTR/snippet problem.")
    if row['low_engagement_time']:
        lines.append("Average time-per-session in the bottom 25% of flagged pages (relative to this pool).")
    return " ".join(lines) if lines else "No specific underperforming signal identified."

df_full_eligible['action'] = df_full_eligible.apply(build_action_text_full, axis=1)


def build_confidence_note_full(row):
    if row['ga4_sessions'] <= 2:
        return "Low confidence — very few GA4 sessions this month, engagement signals (engagement rate, scroll rate, time-per-session) may be unstable."
    if row['gsc_impressions'] < 100:
        return "Moderate confidence — low search impression volume, trend percentage may be noisy."
    return "High confidence — based on adequate impression and session volume across the tracked period."

def build_wrong_note_full(row):
    if row['trend_pct'] < -80:
        return "Could be wrong if this is a 'remontada' case — a sharp recent decline that may already be recovering on the engagement side, which this snapshot wouldn't fully capture."
    if row['ga4_sessions'] <= 2:
        return "Could be wrong if the low GA4 activity reflects a tracking gap rather than genuine underperformance."
    return "Could be wrong if external factors (seasonality, intentional deprioritization by the client) explain the decline rather than a content quality issue."

df_full_eligible['confidence_note'] = df_full_eligible.apply(build_confidence_note_full, axis=1)
df_full_eligible['what_would_make_it_wrong'] = df_full_eligible.apply(build_wrong_note_full, axis=1)


display(df_full_eligible.head(20))


In [ ]:

df_gsc_only_eligible['expected_ctr'] = df_gsc_only_eligible['gsc_sum_position'].apply(lambda x: expected_ctr(x))
low_impressions_threshold = df_gsc_only_eligible['gsc_impressions'].quantile(0.25)
df_gsc_only_eligible['low_impressions'] = df_gsc_only_eligible['gsc_impressions'] < low_impressions_threshold
df_gsc_only_eligible['low_clicks'] = df_gsc_only_eligible['gsc_clicks'] == 0

def build_action_text_gsc(row):
    lines = []
    if row['low_impressions']:
        lines.append("Search impressions in the bottom 25% of flagged pages — visibility problem.")
    if row['low_clicks'] and not row['low_impressions']:
        lines.append("Zero recorded clicks despite adequate impressions — CTR/snippet problem.")
    return " ".join(lines) if lines else "No specific underperforming signal identified."

df_gsc_only_eligible['action'] = df_gsc_only_eligible.apply(build_action_text_gsc, axis=1)


def build_confidence_note_gsc(row):
    if row['gsc_impressions'] < 100:
        return "Low confidence — very low search impression volume, trend percentage and CTR may be noisy."
    if row['gsc_clicks'] == 0:
        return "Moderate confidence — zero clicks recorded, so CTR-based diagnosis has limited signal."
    return "High confidence — based on adequate impression and click volume across the tracked period."

def build_wrong_note_gsc(row):
    if row['trend_pct'] < -80:
        return "Could be wrong if this is a 'remontada' case — a sharp recent decline that may already be recovering, which this snapshot wouldn't capture."
    if row['gsc_impressions'] < 100:
        return "Could be wrong if the low impression count reflects a small/niche keyword rather than a real visibility failure."
    return "Could be wrong if external factors (seasonality, SERP feature changes, intentional deprioritization) explain the decline rather than a content quality issue."

df_gsc_only_eligible['confidence_note'] = df_gsc_only_eligible.apply(build_confidence_note_gsc, axis=1)
df_gsc_only_eligible['what_would_make_it_wrong'] = df_gsc_only_eligible.apply(build_wrong_note_gsc, axis=1)


display(df_gsc_only_eligible.head(20))

In [ ]:
df_ga4_only_eligible['action'] = (
    "Insufficient GA4 traffic (median session count is very low in this tier) to identify a "
    "specific underperforming signal — recommend building visibility/promotion before a content-level refresh."
)

display(df_ga4_only_eligible.head(20))


def build_confidence_note_ga4(row):
    return "Low confidence — this tier's pages have no GSC data and typically very low GA4 traffic (median ~2 sessions), so this flag reflects data scarcity as much as genuine underperformance."

def build_wrong_note_ga4(row):
    return "Could be wrong if this page simply has insufficient tracking history/traffic to judge — more data may reveal it is performing adequately, or the low activity reflects the page's niche rather than a real problem."

df_ga4_only_eligible['confidence_note'] = df_ga4_only_eligible.apply(build_confidence_note_ga4, axis=1)
df_ga4_only_eligible['what_would_make_it_wrong'] = df_ga4_only_eligible.apply(build_wrong_note_ga4, axis=1)

In [ ]:
df_full_eligible.dropna(subset=['score', 'gsc_avg_position', 'ctr', 'gsc_clicks', 'ga4_engaged_sessions', 'scroll_rate', 'avg_engagement_sec_per_session', 'ga4_pageviews', 'ga4_users'], inplace=True)
df_gsc_only_eligible.dropna(subset=['score', 'gsc_sum_position', 'ctr', 'gsc_clicks'], inplace=True)
df_ga4_only_eligible.dropna(subset=['score', 'ga4_engaged_sessions', 'ga4_users', 'avg_engagement_sec_per_session', 'ga4_pageviews', 'scroll_rate'], inplace=True)

full_signal = ['gsc_avg_position', 'ctr', 'gsc_clicks', 'ga4_engaged_sessions', 'scroll_rate', 'avg_engagement_sec_per_session', 'ga4_pageviews', 'ga4_users']
gsc_signal = ['gsc_sum_position', 'ctr', 'gsc_clicks']
ga4_signal = ['ga4_engaged_sessions', 'ga4_users', 'avg_engagement_sec_per_session', 'ga4_pageviews', 'scroll_rate']

In [ ]:
ِِdef model_engine (df):
  for List in [full_signal, gsc_signal, ga4_signal]:
     X = df[List]
     y=df['score']
     groups = df['client_hash_id']
     gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)

     for train_idx, test_idx in gss.split(X, y, groups=groups):
       print("Train:", train_idx, "Test:", test_idx)

       X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
       y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]



## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.